In [ ]:
from pathlib import Path
import pandas as pd
from pesto_tracker import PESTOTracker
import numpy as np

In [ ]:
STEP_SIZE =0.01
SAMPLE_RATE =44100
f0Pesto_tracker = PESTOTracker(use_gpu=False, step_size=STEP_SIZE, sample_rate=SAMPLE_RATE)

stems_folder = Path("../SourceSeparation/demucs_output")
wav_files = list(stems_folder.glob("**/voice.wav"))
print(f"{len(wav_files)} vocal stems found.")

f0_automatic_annotations_folder = Path("../data/cante2midi_f0/")
autogt_f0_files = list(f0_automatic_annotations_folder.glob("*.csv"))

f0_groundtruth_folder = Path("../data/cante2midi_groundTruth")
gt_f0_files = list(f0_groundtruth_folder.glob("*.csv"))

paired_data = {}

f0_tracking=True
for file in wav_files:
    print(file)
    file_name= file.parent.stem
    idx = file_name.split('_')[0]
    gt_file = list(f0_groundtruth_folder.glob(f"{file_name}.notes"))[0]
    autogt_file = list(f0_automatic_annotations_folder.glob(f"{file_name}.f0.csv"))[0]


    paired_data[idx]= {'stem':file,
                       'gt':gt_file,
                       'autogt':autogt_file,
                       
                    }
    if f0_tracking:
        output_path = "f0Contour_PESTO/"+file_name+".f0.csv"
        f0Pesto_tracker.save_pitch_contour(audio_path=file,
                                           output_path=output_path)
        
        final_pitches, timesteps, confidence = f0Pesto_tracker.extract_f0(file)
        paired_data[idx]['PESTO']= {
                           'pitch':final_pitches,
                           'timesteps':timesteps,
                           'confidence':confidence
                       }
    


In [ ]:
def noteToFreq(note):
    a = 440 #frequency of A (coomon value is 440Hz)
    return (a / 32) * (2 ** ((note - 9) / 12))

In [ ]:
annotations_05 = pd.read_csv(paired_data['05']['gt'], sep=",", names = ['onset_time', 'duration', 'midinote', 'vel'], index_col=False)
annotations_05['freq'] = annotations_05.midinote.apply(lambda x: noteToFreq(x))

auto_annotations_05 = pd.read_csv(paired_data['05']['autogt'], names = ['timestamp', 'freq'])
auto_annotations_05["freq"] = auto_annotations_05["freq"].clip(lower=0)

predicted_05 = pd.DataFrame(data={
    'timestamp':paired_data['05']['PESTO']['timesteps']/1000,
    'freq': paired_data['05']['PESTO']['pitch'],
    'confidence':paired_data['05']['PESTO']['confidence']
})

print(f"Annotated landmarks: {len(annotations_05)}")
print(f"Automatically annotatd f0: {len(auto_annotations_05)}")
print(f"PESTO detected f0: {len(predicted_05)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Example: load your CSVs
# ref_df = pd.read_csv("reference.csv")  # columns: timestamp, freq
# pred_df = pd.read_csv("prediction.csv")  # columns: timestamp, freq

# Replace -220 with NaN for plotting
annotations_05['freq_plot'] = annotations_05['freq'].replace(-220.0, np.nan)
predicted_05['freq_plot'] = predicted_05['freq'].replace(0.0, np.nan)
auto_annotations_05['freq_plot'] = auto_annotations_05['freq'].replace(0, np.nan)

# Create figure
plt.figure(figsize=(12, 4))

# Plot reference
plt.plot(annotations_05['onset_time'], annotations_05['freq_plot'], label='Reference', color='blue', linewidth=1)
plt.plot(auto_annotations_05['timestamp'], auto_annotations_05['freq'], label='Auto-Annotations', color='green', alpha=0.5, linewidth=1)
# Plot predictions
plt.plot(predicted_05['timestamp'], predicted_05['freq_plot'], label='Prediction', color='orange', alpha=0.7, linewidth=1)

# Labels
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Pitch Comparison: Reference vs Prediction (Track 05)')
plt.legend()
plt.grid(True)
#plt.xlim(0, 5)  # first 5 seconds
plt.show()

In [ ]:
annotations_08 = pd.read_csv(paired_data['08']['gt'], sep=",", names = ['onset_time', 'duration', 'midinote', 'vel'], index_col=False)
annotations_08['freq'] = annotations_08.midinote.apply(lambda x: noteToFreq(x))

auto_annotations_08 = pd.read_csv(paired_data['08']['autogt'], names = ['timestamp', 'freq'])
auto_annotations_08["freq"] = auto_annotations_08["freq"].clip(lower=0)

predicted_08 = pd.DataFrame(data={
    'timestamp':paired_data['08']['PESTO']['timesteps']/1000,#STEP_SIZE/(STEP_SIZE*1000),
    'freq': paired_data['08']['PESTO']['pitch'],
    'confidence':paired_data['08']['PESTO']['confidence']
})

print(f"Annotated landmarks: {len(annotations_08)}")
print(f"Automatically annotatd f0: {len(auto_annotations_08)}")
print(f"PESTO detected f0: {len(predicted_08)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Example: load your CSVs
# ref_df = pd.read_csv("reference.csv")  # columns: timestamp, freq
# pred_df = pd.read_csv("prediction.csv")  # columns: timestamp, freq

# Replace -220 with NaN for plotting
annotations_08['freq_plot'] = annotations_08['freq'].replace(-220.0, np.nan)
predicted_08['freq_plot'] = predicted_08['freq'].replace(0, np.nan)
auto_annotations_08['freq_plot'] = auto_annotations_08['freq'].replace(0, np.nan)

# Create figure
plt.figure(figsize=(12, 4))

# Plot reference
plt.plot(annotations_08['onset_time'], annotations_08['freq_plot'], label='Reference', color='blue', linewidth=1, marker='o', markersize=3)
plt.plot(auto_annotations_08['timestamp'], auto_annotations_08['freq'], label='Auto-Annotations', color='green', alpha=0.5, linewidth=1)
# Plot predictions
plt.plot(predicted_08['timestamp'], predicted_08['freq_plot'], label='Prediction', color='orange', alpha=0.7, linewidth=1)

# Labels
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Pitch Comparison: Reference vs Prediction (Track 08)')
plt.legend()
plt.grid(True)
#plt.xlim(0, 5)  # first 5 seconds
plt.show()